# Phase 2 — Activity 2: SegResNet Fine-tuning & Embedding Extraction

**Project:** Explainable Disease Progression · Cyprus PROTEAS Dataset  
**Model:** MONAI SegResNet with **BraTS pretrained weights** (fine-tuned)  
**Labels:** 3-region BraTS format: TC (Tumor Core), WT (Whole Tumor), ET (Enhancing Tumor)  
**Environment:** Kaggle T4 GPU (15GB VRAM)  

---

## Why Pretrained?
Training from scratch with only 114 samples gives Dice ~0.09.  
With BraTS pretrained weights (trained on 1000+ cases), we expect **Dice 0.50–0.70** after fine-tuning.

## Modes
| Mode | Duration | Purpose |
|------|----------|--------|
| `quick_test` | ~30 min | Validate pipeline: 5 epochs on fold 0 |
| `train` | ~4 hrs/fold | Full training: 50 epochs with early stopping |
| `train_all` | ~8 hrs | Train all 3 folds + extract embeddings |


---
## 0. Configuration

In [1]:
# ╔════════════════════════════════════════════════════════════╗
# ║  CHANGE THESE FOR EACH SESSION                           ║
# ╚════════════════════════════════════════════════════════════╝

MODE = 'train_all'   # 'quick_test' | 'train' | 'train_all'
FOLD = 0              # 0, 1, or 2 (ignored if train_all)
CV_TYPE = '3fold'
RESUME_PATH = None

# ╔════════════════════════════════════════════════════════════╗
# ║  TRAINING HYPERPARAMETERS                                ║
# ╚════════════════════════════════════════════════════════════╝

CONFIG = {
    # Model (SegResNet — must match pretrained weights)
    'in_channels': 4,
    'out_channels': 3,           # TC, WT, ET (3 binary regions)
    'init_filters': 16,          # Must match pretrained weights!
    'blocks_down': (1, 2, 2, 4),
    'blocks_up': (1, 1, 1),
    'dropout_prob': 0.2,
    
    # Training
    'lr': 1e-5,              # Low LR for fine-tuning
    'weight_decay': 1e-5,
    'cache_rate': 0.5,
}

if MODE == 'quick_test':
    CONFIG['epochs'] = 5
    CONFIG['patience'] = 5
    CONFIG['val_interval'] = 1
    CONFIG['resolution'] = [96, 96, 96]
    CONFIG['batch_size'] = 2
    CONFIG['num_workers'] = 0       # MUST be 0 — Kaggle symlink race condition
else:
    CONFIG['epochs'] = 50
    CONFIG['patience'] = 15
    CONFIG['val_interval'] = 2
    CONFIG['resolution'] = [128, 128, 128]
    CONFIG['batch_size'] = 2       # batch=2 fits T4 with SegResNet(16)
    CONFIG['num_workers'] = 0       # MUST be 0 — Kaggle symlink race condition

REGION_NAMES = ['TC', 'WT', 'ET']

print(f'Mode: {MODE} | Fold: {FOLD} | CV: {CV_TYPE}')
print(f'Epochs: {CONFIG["epochs"]} | LR: {CONFIG["lr"]} | Res: {CONFIG["resolution"]} | Batch: {CONFIG["batch_size"]}')
if MODE == 'train_all':
    print(f'Training ALL 3 folds + embeddings (~8 hours)')


Mode: train_all | Fold: 0 | CV: 3fold
Epochs: 50 | LR: 1e-05 | Res: [128, 128, 128] | Batch: 2
Training ALL 3 folds + embeddings (~8 hours)


---
## 1. Install Dependencies & Setup

In [2]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai>=1.4.0', '--quiet'])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 44.8 MB/s eta 0:00:00


0

In [3]:
import os, json, time, glob
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn
from torch.amp import GradScaler, autocast

import monai
from monai.networks.nets import SegResNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    Orientationd, Spacingd, CropForegroundd, Resized,
    RandFlipd, RandRotate90d, RandScaleIntensityd,
    RandShiftIntensityd, RandGaussianNoised, RandGaussianSmoothd,
    RandAffined, RandAdjustContrastd,
    NormalizeIntensityd, MapTransform,
)
from monai.data import CacheDataset, DataLoader
from monai.inferers import sliding_window_inference

print(f'PyTorch: {torch.__version__}')
print(f'MONAI:   {monai.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2026-03-29 16:15:41.295594: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774800941.518764      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774800941.581127      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774800942.101452      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774800942.101495      25 computation_placer.cc:1

PyTorch: 2.10.0+cu128
MONAI:   1.5.2
CUDA:    True
GPU:     Tesla T4
VRAM:    15.6 GB


In [4]:
# Auto-detect environment
KAGGLE_PATHS = [
    Path('/kaggle/input/cyprus-proteas-brain-mets'),
    Path('/kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets'),
    Path('/kaggle/input/cyprus-proteas'),
]
LOCAL_DATA = Path('/home/moamed/canada_me/explainable_diseas/implementation_cyprus/Data/Cyprus-PROTEAS-zips')

DATA_ROOT = None
ENV = 'local'
for p in KAGGLE_PATHS:
    if p.exists():
        DATA_ROOT = p
        ENV = 'kaggle'
        break
if DATA_ROOT is None:
    DATA_ROOT = LOCAL_DATA

OUTPUT_ROOT = Path('/kaggle/working/phase2_outputs') if ENV == 'kaggle' else Path('/home/moamed/canada_me/explainable_diseas/implementation_cyprus/outputs/phase2_outputs')
for subdir in ['checkpoints', 'metrics', 'embeddings', 'figures']:
    (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)

SPLITS_FILE = None
for candidate in [
    DATA_ROOT / 'data_splits.json',
    Path('/home/moamed/canada_me/explainable_diseas/implementation_cyprus/outputs/data_splits.json'),
    Path('/kaggle/input/cyprus-proteas-brain-mets/data_splits.json'),
    Path('/kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets/data_splits.json'),
]:
    if candidate.exists():
        SPLITS_FILE = candidate
        break

print(f'Environment:  {ENV}')
print(f'Data root:    {DATA_ROOT}')
print(f'Output root:  {OUTPUT_ROOT}')
print(f'Splits file:  {SPLITS_FILE}')
assert SPLITS_FILE is not None, 'data_splits.json not found!'


Environment:  kaggle
Data root:    /kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets
Output root:  /kaggle/working/phase2_outputs
Splits file:  /kaggle/input/datasets/mohamedmohamed23/cyprus-proteas-brain-mets/data_splits.json


### 1.1 Fix File Extensions (Kaggle)

In [5]:
# Fix .nii_gz → .nii.gz via symlinks (Kaggle decompresses .gz files)
nii_gz_files = glob.glob(str(DATA_ROOT / '**/*.nii_gz'), recursive=True)

if nii_gz_files:
    SYMLINK_ROOT = Path('/kaggle/working/data')
    print(f'Found {len(nii_gz_files)} .nii_gz files — creating symlinks...')
    for src in nii_gz_files:
        src = Path(src)
        rel = src.relative_to(DATA_ROOT)
        dst = SYMLINK_ROOT / str(rel).replace('.nii_gz', '.nii.gz')
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.symlink(src, dst)
    for ext in ['*.json', '*.xlsx', '*.csv']:
        for src in DATA_ROOT.glob(ext):
            dst = SYMLINK_ROOT / src.name
            if not dst.exists():
                os.symlink(src, dst)
    DATA_ROOT = SYMLINK_ROOT
    sf = SYMLINK_ROOT / 'data_splits.json'
    if sf.exists():
        SPLITS_FILE = sf
    print(f'DATA_ROOT → {DATA_ROOT} ({len(list(SYMLINK_ROOT.rglob("*.nii.gz")))} symlinks)')
else:
    print('No .nii_gz files — data is .nii.gz format ✅')


Found 1050 .nii_gz files — creating symlinks...
DATA_ROOT → /kaggle/working/data (1050 symlinks)


---
## 2. Load Data & Label Conversion

### Label Remapping
Cyprus PROTEAS uses labels `{0, 1, 2, 3}` but BraTS uses `{0, 1, 2, 4}`.  
We remap **label 3 → 4** then convert to **3 binary region channels**: TC, WT, ET.


In [6]:
with open(SPLITS_FILE) as f:
    all_splits = json.load(f)


def resolve_path(data_root, rel_path):
    """Resolve path handling .nii.gz / .nii_gz / .nii variants."""
    full = data_root / rel_path
    if full.exists() and full.stat().st_size > 0:
        return str(full)
    if str(full).endswith('.nii.gz'):
        nii_gz = Path(str(full)[:-7] + '.nii_gz')
        if nii_gz.exists() and nii_gz.stat().st_size > 0:
            return str(nii_gz)
    if str(full).endswith('.gz'):
        no_gz = Path(str(full)[:-3])
        if no_gz.exists() and no_gz.stat().st_size > 0:
            return str(no_gz)
    raise FileNotFoundError(f'Cannot find: {rel_path}')


def validate_3d(filepath):
    import nibabel as nib
    try:
        shape = nib.load(filepath).shape
        if len(shape) < 3 or any(s < 10 for s in shape[:3]):
            return False
        return True
    except Exception:
        return False


def scans_to_monai(scan_list):
    dicts, skipped = [], []
    for scan in scan_list:
        try:
            paths = {k: resolve_path(DATA_ROOT, scan[k]) for k in ['t1', 't1c', 't2', 'fla', 'mask']}
            if not all(validate_3d(paths[k]) for k in paths):
                skipped.append(f"{scan['patient_dir']}/{scan['visit']}: 2D file")
                continue
            dicts.append({
                'image': [paths['t1'], paths['t1c'], paths['t2'], paths['fla']],
                'label': paths['mask'],
                'patient_dir': scan['patient_dir'],
                'visit': scan['visit'],
                'patient_group': scan['patient_group'],
            })
        except FileNotFoundError as e:
            skipped.append(str(e))
    if skipped:
        print(f'  Skipped {len(skipped)} scans')
    return dicts


print('Splits loaded ✅')


Splits loaded ✅


In [7]:
# ── Custom transform: Cyprus labels → BraTS 3-region format ──

class ConvertCyprusToBratsRegionsd(MapTransform):
    """
    Convert Cyprus PROTEAS labels (0,1,2,3) to BraTS 3-region binary channels.
    MUST be the LAST transform — creates new tensor that loses MONAI metadata.
    """
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            label = d[key]
            label_r = label.clone()
            label_r[label == 3] = 4
            tc = torch.logical_or(label_r == 1, label_r == 4).float()
            wt = torch.logical_or(torch.logical_or(label_r == 1, label_r == 2), label_r == 4).float()
            et = (label_r == 4).float()
            d[key] = torch.cat([tc, wt, et], dim=0)
        return d


res = CONFIG['resolution']

# ═══════════════════════════════════════════════════════════════
#  STRONG augmentation pipeline (critical for 114 training samples)
#  Inspired by nnU-Net + BraTS best practices
# ═══════════════════════════════════════════════════════════════

train_transforms = Compose([
    # ── Loading & preprocessing ──
    LoadImaged(keys=['image', 'label'], image_only=False),
    EnsureChannelFirstd(keys=['label']),
    EnsureTyped(keys=['image', 'label']),
    Orientationd(keys=['image', 'label'], axcodes='RAS'),
    Spacingd(keys=['image', 'label'], pixdim=(1.0, 1.0, 1.0), mode=('bilinear', 'nearest')),
    CropForegroundd(keys=['image', 'label'], source_key='image', margin=5),
    Resized(keys=['image', 'label'], spatial_size=res, mode=('trilinear', 'nearest')),
    NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True),
    
    # ── Spatial augmentation (nnU-Net style) ──
    RandAffined(
        keys=['image', 'label'],
        prob=0.3,
        rotate_range=(0.26, 0.26, 0.26),  # ±15 degrees
        scale_range=(0.1, 0.1, 0.1),       # ±10% scaling
        mode=('bilinear', 'nearest'),
        padding_mode='border',
    ),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=0),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=1),
    RandFlipd(keys=['image', 'label'], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=['image', 'label'], prob=0.3, spatial_axes=(0, 1)),
    
    # ── Intensity augmentation ──
    RandScaleIntensityd(keys=['image'], factors=0.1, prob=0.5),
    RandShiftIntensityd(keys=['image'], offsets=0.1, prob=0.5),
    RandGaussianNoised(keys=['image'], std=0.05, prob=0.2),
    RandGaussianSmoothd(keys=['image'], sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0), prob=0.2),
    RandAdjustContrastd(keys=['image'], gamma=(0.7, 1.5), prob=0.3),
    
    # ── Label conversion (MUST be last) ──
    ConvertCyprusToBratsRegionsd(keys=['label']),
    EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

val_transforms = Compose([
    LoadImaged(keys=['image', 'label'], image_only=False),
    EnsureChannelFirstd(keys=['label']),
    EnsureTyped(keys=['image', 'label']),
    Orientationd(keys=['image', 'label'], axcodes='RAS'),
    Spacingd(keys=['image', 'label'], pixdim=(1.0, 1.0, 1.0), mode=('bilinear', 'nearest')),
    CropForegroundd(keys=['image', 'label'], source_key='image', margin=5),
    Resized(keys=['image', 'label'], spatial_size=res, mode=('trilinear', 'nearest')),
    NormalizeIntensityd(keys=['image'], nonzero=True, channel_wise=True),
    # No augmentation for validation!
    ConvertCyprusToBratsRegionsd(keys=['label']),
    EnsureTyped(keys=['image', 'label'], dtype=torch.float32),
])

print('Transforms defined ✅')
print('  STRONG augmentation enabled:')
print('    ✅ RandAffine (rotation ±15°, scale ±10%)')
print('    ✅ RandFlip (3 axes)')
print('    ✅ RandRotate90')
print('    ✅ RandScaleIntensity + RandShiftIntensity')
print('    ✅ RandGaussianNoise + RandGaussianSmooth')
print('    ✅ RandAdjustContrast')


Transforms defined ✅
  STRONG augmentation enabled:
    ✅ RandAffine (rotation ±15°, scale ±10%)
    ✅ RandFlip (3 axes)
    ✅ RandRotate90
    ✅ RandScaleIntensity + RandShiftIntensity
    ✅ RandGaussianNoise + RandGaussianSmooth
    ✅ RandAdjustContrast


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


---
## 3. Download Pretrained SegResNet (BraTS)

In [8]:
# Download MONAI BraTS pretrained bundle
BUNDLE_DIR = OUTPUT_ROOT / 'models'
BUNDLE_DIR.mkdir(exist_ok=True)

bundle_path = BUNDLE_DIR / 'brats_mri_segmentation'
if not bundle_path.exists():
    print('Downloading BraTS pretrained SegResNet...')
    from monai.bundle import download
    download(name='brats_mri_segmentation', bundle_dir=str(BUNDLE_DIR))
    print('Downloaded ✅')
else:
    print(f'Bundle already exists at {bundle_path} ✅')

# Find the pretrained weights file
pretrained_weights = None
for candidate in [
    bundle_path / 'models' / 'model.pt',
    bundle_path / 'models' / 'model.pth',
]:
    if candidate.exists():
        pretrained_weights = candidate
        break

if pretrained_weights:
    print(f'Pretrained weights: {pretrained_weights} ({pretrained_weights.stat().st_size/1e6:.1f} MB)')
else:
    import os
    print('Available files in bundle:')
    for root, dirs, files in os.walk(bundle_path):
        for f in files:
            fp = os.path.join(root, f)
            print(f'  {os.path.relpath(fp, bundle_path)}')


2026-03-29 16:15:56,630 - INFO - --- input summary of monai.bundle.scripts.download ---
2026-03-29 16:15:56,631 - INFO - > name: 'brats_mri_segmentation'
2026-03-29 16:15:56,632 - INFO - > bundle_dir: '/kaggle/working/phase2_outputs/models'
2026-03-29 16:15:56,632 - INFO - > source: 'monaihosting'
2026-03-29 16:15:56,633 - INFO - > remove_prefix: 'monai_'
2026-03-29 16:15:56,634 - INFO - > progress: True
2026-03-29 16:15:56,634 - INFO - ---




Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Downloaded ✅
Pretrained weights: /kaggle/working/phase2_outputs/models/brats_mri_segmentation/models/model.pt (18.8 MB)


---
## 4. Training Functions

All training logic is wrapped in a function so we can loop over folds.

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def create_model():
    """Create SegResNet and load pretrained BraTS weights."""
    model = SegResNet(
        spatial_dims=3,
        in_channels=CONFIG['in_channels'],
        out_channels=CONFIG['out_channels'],
        init_filters=CONFIG['init_filters'],
        blocks_down=CONFIG['blocks_down'],
        blocks_up=CONFIG['blocks_up'],
        dropout_prob=CONFIG['dropout_prob'],
    ).to(device)
    
    # Load pretrained weights
    if pretrained_weights:
        ckpt = torch.load(pretrained_weights, map_location=device, weights_only=False)
        # Handle different checkpoint formats
        if isinstance(ckpt, dict) and 'state_dict' in ckpt:
            model.load_state_dict(ckpt['state_dict'], strict=False)
        elif isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
            model.load_state_dict(ckpt['model_state_dict'], strict=False)
        else:
            model.load_state_dict(ckpt, strict=False)
        print('  Loaded BraTS pretrained weights ✅')
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  SegResNet: {n_params:,} parameters')
    return model


def find_bottleneck(model, target_channels=None, resolution=(96,96,96)):
    """Auto-discover bottleneck layer by running test forward pass."""
    candidates = []
    handles = []
    for name, module in model.named_modules():
        if not name:
            continue
        def make_hook(n, mod):
            def hook(m, inp, out):
                if isinstance(out, torch.Tensor) and out.dim() == 5:
                    candidates.append({'name': n, 'module': mod,
                        'channels': out.shape[1], 'spatial': out.shape[2]*out.shape[3]*out.shape[4]})
            return hook
        handles.append(module.register_forward_hook(make_hook(name, module)))
    
    with torch.no_grad():
        _ = model(torch.randn(1, 4, *resolution).to(device))
    for h in handles:
        h.remove()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    if target_channels:
        filtered = [c for c in candidates if c['channels'] == target_channels]
    else:
        max_ch = max(c['channels'] for c in candidates)
        filtered = [c for c in candidates if c['channels'] == max_ch]
    min_s = min(c['spatial'] for c in filtered)
    return [c for c in filtered if c['spatial'] == min_s][-1]


# Test model creation
test_model = create_model()
bn = find_bottleneck(test_model, resolution=tuple(CONFIG['resolution']))
print(f'  Bottleneck: {bn["name"]} ({bn["channels"]} channels)')
del test_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


  Loaded BraTS pretrained weights ✅
  SegResNet: 4,702,227 parameters
  Bottleneck: down_layers.3 (128 channels)


In [10]:
def train_fold(fold, resume_path=None):
    """Train one fold: returns (model, best_dice, metrics)"""
    print(f'\n{"="*60}')
    print(f'  Training Fold {fold} | {MODE.upper()} | {CONFIG["epochs"]} epochs')
    print(f'{"="*60}\n')
    
    # Data
    fold_data = all_splits[CV_TYPE][f'fold_{fold}']
    train_dicts = scans_to_monai(fold_data['train_scans'])
    val_dicts = scans_to_monai(fold_data['test_scans'])
    print(f'  Train: {len(train_dicts)} | Val: {len(val_dicts)}')
    
    # Pre-filter: remove scans that fail to load as 3D
    def validate_3d(dicts):
        import nibabel as nib
        valid = []
        for d in dicts:
            try:
                # Check mask file is 3D
                mask_path = d['label']
                hdr = nib.load(mask_path).header
                shape = hdr.get_data_shape()
                if len(shape) >= 3 and shape[2] > 1:
                    valid.append(d)
                else:
                    pid = d.get('patient_dir', '?')
                    print(f'  ⚠️ Skipped 2D file: {pid} (shape={shape})')
            except Exception as e:
                pid = d.get('patient_dir', '?')
                print(f'  ⚠️ Skipped bad file: {pid} ({e})')
        return valid
    
    train_dicts = validate_3d(train_dicts)
    val_dicts = validate_3d(val_dicts)
    print(f'  After 3D validation: Train={len(train_dicts)} | Val={len(val_dicts)}')
    
    train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=CONFIG['cache_rate'], num_workers=CONFIG['num_workers'])
    val_ds = CacheDataset(val_dicts, val_transforms, cache_rate=1.0, num_workers=CONFIG['num_workers'])
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'], pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)
    
    # Model
    model = create_model()
    
    # Loss (sigmoid for binary multi-channel)
    loss_fn = DiceLoss(sigmoid=True, smooth_nr=0, smooth_dr=1e-5, squared_pred=True, batch=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    scaler = GradScaler('cuda')
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    
    # Resume
    start_epoch, best_dice = 0, 0.0
    train_losses, val_dices = [], []
    epochs_no_improve = 0
    
    if resume_path and Path(resume_path).exists():
        ckpt = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_dice = ckpt.get('best_dice', 0.0)
        print(f'  Resumed from epoch {start_epoch}, best Dice: {best_dice:.4f}')
    
    # Train
    t_start = time.time()
    
    for epoch in range(start_epoch, CONFIG['epochs']):
        model.train()
        epoch_loss, n_steps = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(device)
            labels = batch['label'].to(device)
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(images)
                loss = loss_fn(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()
            n_steps += 1
        scheduler.step()
        avg_loss = epoch_loss / max(n_steps, 1)
        train_losses.append(avg_loss)
        
        # Validate
        if (epoch + 1) % CONFIG['val_interval'] == 0 or epoch == CONFIG['epochs'] - 1:
            model.eval()
            dice_metric.reset()
            with torch.no_grad():
                for val_batch in val_loader:
                    val_img = val_batch['image'].to(device)
                    val_lbl = val_batch['label'].to(device)
                    with autocast('cuda'):
                        val_out = sliding_window_inference(val_img, CONFIG['resolution'], sw_batch_size=1, predictor=model)
                    val_pred = (torch.sigmoid(val_out) > 0.5).float()
                    dice_metric(val_pred, val_lbl)
            
            dice_values = dice_metric.aggregate().cpu().numpy().tolist()
            mean_dice = np.mean(dice_values)
            elapsed = (time.time() - t_start) / 60
            
            val_dices.append({'epoch': epoch, 'mean_dice': mean_dice,
                'dice_TC': dice_values[0], 'dice_WT': dice_values[1], 'dice_ET': dice_values[2]})
            
            print(f'Epoch {epoch:3d}/{CONFIG["epochs"]-1} | Loss: {avg_loss:.4f} | '
                  f'Dice: {mean_dice:.4f} (TC={dice_values[0]:.3f} WT={dice_values[1]:.3f} ET={dice_values[2]:.3f}) | {elapsed:.1f} min')
            
            if mean_dice > best_dice:
                best_dice = mean_dice
                epochs_no_improve = 0
                torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'best_dice': best_dice, 'config': CONFIG,
                    'train_losses': train_losses, 'val_dices': val_dices, 'fold': fold},
                    OUTPUT_ROOT / f'checkpoints/segresnet_fold{fold}_best.pth')
                print(f'  ✅ New best! Saved.')
            else:
                epochs_no_improve += CONFIG['val_interval']
        
        # Save last checkpoint
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_dice': best_dice, 'config': CONFIG,
            'train_losses': train_losses, 'val_dices': val_dices, 'fold': fold},
            OUTPUT_ROOT / f'checkpoints/segresnet_fold{fold}_last.pth')
        
        if epochs_no_improve >= CONFIG['patience']:
            print(f'  Early stopping after {CONFIG["patience"]} epochs without improvement')
            break
    
    total_time = (time.time() - t_start) / 60
    print(f'\n  Fold {fold} complete: Best Dice = {best_dice:.4f} | Time = {total_time:.1f} min')
    
    # Save metrics
    metrics = {'fold': fold, 'mode': MODE, 'best_dice': best_dice, 'total_time_minutes': total_time,
        'train_losses': train_losses, 'val_dices': val_dices, 'config': CONFIG,
        'timestamp': datetime.now().isoformat()}
    with open(OUTPUT_ROOT / f'metrics/fold{fold}_metrics.json', 'w') as f:
        json.dump(metrics, f, indent=2)
    
    return model, best_dice, metrics

print('Training function defined ✅')


Training function defined ✅


In [11]:
def extract_embeddings(fold, model):
    """Extract bottleneck embeddings for all scans in a fold."""
    print(f'\n  Extracting embeddings for fold {fold}...')
    
    fold_data = all_splits[CV_TYPE][f'fold_{fold}']
    all_dicts, is_test_flags = [], []
    
    for is_test, scan_list in [(False, fold_data['train_scans']), (True, fold_data['test_scans'])]:
        for scan in scan_list:
            try:
                paths = {k: resolve_path(DATA_ROOT, scan[k]) for k in ['t1', 't1c', 't2', 'fla', 'mask']}
                if not all(validate_3d(paths[k]) for k in paths):
                    continue
                all_dicts.append({'image': [paths['t1'], paths['t1c'], paths['t2'], paths['fla']],
                    'label': paths['mask'], 'patient_dir': scan['patient_dir'], 'visit': scan['visit']})
                is_test_flags.append(is_test)
            except FileNotFoundError:
                continue
    
    emb_ds = CacheDataset(all_dicts, val_transforms, cache_rate=1.0, num_workers=0)
    emb_loader = DataLoader(emb_ds, batch_size=1, shuffle=False, num_workers=0)
    
    # Find and hook bottleneck
    model.eval()
    bn = find_bottleneck(model, resolution=tuple(CONFIG['resolution']))
    emb_dim = bn['channels']
    
    features = []
    def hook_fn(mod, inp, out):
        features.append(out.detach())
    handle = bn['module'].register_forward_hook(hook_fn)
    
    embeddings = {}
    with torch.no_grad():
        for i, batch in enumerate(emb_loader):
            features.clear()
            with autocast('cuda'):
                _ = model(batch['image'].to(device))
            if features:
                emb = features[0].mean(dim=(-3, -2, -1)).cpu().numpy().squeeze()
                key = f"{all_dicts[i]['patient_dir']}_{all_dicts[i]['visit']}"
                embeddings[key] = emb
            if (i + 1) % 25 == 0:
                print(f'    {i+1}/{len(emb_loader)} processed')
    
    handle.remove()
    
    # Save
    if embeddings:
        np.savez(OUTPUT_ROOT / f'embeddings/cnn_embeddings_fold{fold}.npz', **embeddings)
        meta = {k: {'patient_dir': k.rsplit('_',1)[0], 'visit': k.rsplit('_',1)[1],
            'is_test': is_test_flags[i], 'fold': fold} for i, k in enumerate(embeddings.keys())}
        with open(OUTPUT_ROOT / f'embeddings/cnn_embeddings_fold{fold}_meta.json', 'w') as f:
            json.dump(meta, f, indent=2)
        print(f'  ✅ {len(embeddings)} embeddings × {emb_dim}-dim saved')
    else:
        print('  ❌ No embeddings extracted!')
    
    return embeddings, emb_dim

print('Embedding extraction function defined ✅')


Embedding extraction function defined ✅


---
## 5. Run Training + Extraction

In [12]:
# ── Main execution ──

if MODE == 'train_all':
    folds = [0, 1, 2]
else:
    folds = [FOLD]

results = {}

for fold in folds:
    # Train
    model, best_dice, metrics = train_fold(fold, resume_path=RESUME_PATH)
    
    # Load best model for extraction
    best_path = OUTPUT_ROOT / f'checkpoints/segresnet_fold{fold}_best.pth'
    if best_path.exists():
        ckpt = torch.load(best_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
    
    # Extract embeddings
    embeddings, emb_dim = extract_embeddings(fold, model)
    
    results[fold] = {'best_dice': best_dice, 'n_embeddings': len(embeddings), 'emb_dim': emb_dim}
    
    # Free GPU memory before next fold
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Summary ──
print(f'\n{"="*60}')
print(f'  SESSION COMPLETE')
print(f'{"="*60}')
for fold, r in results.items():
    print(f'  Fold {fold}: Dice={r["best_dice"]:.4f} | {r["n_embeddings"]} embeddings × {r["emb_dim"]}-dim')

print(f'\n  Files saved to {OUTPUT_ROOT}:')
for f in sorted(OUTPUT_ROOT.rglob('*')):
    if f.is_file():
        print(f'    {f.relative_to(OUTPUT_ROOT)} ({f.stat().st_size/1024:.1f} KB)')
print(f'\n  ⚠️ DOWNLOAD these files before session ends!')



  Training Fold 0 | TRAIN_ALL | 50 epochs

  Train: 114 | Val: 56
  After 3D validation: Train=114 | Val=56



Loading dataset: 100%|██████████| 57/57 [01:22<00:00,  1.44s/it]

Loading dataset: 100%|██████████| 56/56 [01:17<00:00,  1.38s/it]


  Loaded BraTS pretrained weights ✅
  SegResNet: 4,702,227 parameters


/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out[idx_zm] += p


Epoch   1/49 | Loss: 0.8696 | Dice: 0.1774 (TC=0.146 WT=0.306 ET=0.080) | 4.9 min
  ✅ New best! Saved.


...

Epoch  47/49 | Loss: 0.4843 | Dice: 0.3625 (TC=0.363 WT=0.383 ET=0.342) | 86.4 min


/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:231: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  win_data = inputs[unravel_slice[0]].to(sw_device)
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:370: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  out[idx_zm] += p


Epoch  49/49 | Loss: 0.4970 | Dice: 0.3625 (TC=0.363 WT=0.383 ET=0.342) | 90.1 min

  Fold 2 complete: Best Dice = 0.3634 | Time = 90.1 min

  Extracting embeddings for fold 2...



Loading dataset: 100%|██████████| 170/170 [03:36<00:00,  1.27s/it]


    25/170 processed
    50/170 processed
    75/170 processed
    100/170 processed
    125/170 processed
    150/170 processed
  ✅ 170 embeddings × 128-dim saved

  SESSION COMPLETE
  Fold 0: Dice=0.4134 | 170 embeddings × 128-dim
  Fold 1: Dice=0.3273 | 170 embeddings × 128-dim
  Fold 2: Dice=0.3634 | 170 embeddings × 128-dim

  Files saved to /kaggle/working/phase2_outputs:
    checkpoints/segresnet_fold0_best.pth (55213.4 KB)
    checkpoints/segresnet_fold0_last.pth (55214.0 KB)
    checkpoints/segresnet_fold1_best.pth (55212.7 KB)
    checkpoints/segresnet_fold1_last.pth (55213.8 KB)
    checkpoints/segresnet_fold2_best.pth (55213.6 KB)
    checkpoints/segresnet_fold2_last.pth (55214.0 KB)
    embeddings/cnn_embeddings_fold0.npz (83.9 KB)
    embeddings/cnn_embeddings_fold0_meta.json (17.4 KB)
    embeddings/cnn_embeddings_fold1.npz (83.9 KB)
    embeddings/cnn_embeddings_fold1_meta.json (17.4 KB)
    embeddings/cnn_embeddings_fold2.npz (83.9 KB)
    embeddings/cnn_embeddings_fol